# كشكول التجارب: متى نستخدم ومتى نتجنب تقنية MMR؟ (المحاضرة 48)

أهلا بك في كشكول التجارب التفاعلي لمحاضرة 48.
في هذا الدرس، سنختبر عمليا القاعدة الهندسية الأهم للمقابلات وأنظمة الإنتاج:
متى يكون مسترجع MMR هو البطل لمنع التكرار؟ ومتى نتجنبه تماما ونعتمد على البحث التقليدي؟

### 1. تجهيز المكتبات وبيانات تجربة LangChain
زي ما شرح المحاضر، تعال نجهز مجموعة وثائق عن إطار عمل LangChain فيها وثائق مكررة عن فهارس الفيكتور، ووثائق تانية عن الوكلاء، الذاكرة، وسلاسل الأوامر:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docs = [
    Document(page_content="يتيح LangChain ربط النماذج بمخازن متجهات FAISS للبحث السريع عن أقرب المتجهات الدلالية."),
    Document(page_content="استخدام FAISS مع LangChain يساعد في فهرسة الوثائق الكثيفة والبحث السريع عن المستندات الشبيهة."),
    Document(page_content="فهارس FAISS في لانج تشين تسرع البحث الدلالي في المشاريع الكبرى."),
    Document(page_content="يوفر LangChain دعم الوكلاء Agents الذين يمتلكون أدوات Tools لتنفيذ مهام الحساب والبحث على الويب."),
    Document(page_content="إدارة الذاكرة Memory في LangChain تمكن الشات بوت من تذكر سياق الحوار عبر الأدوار المختلفة."),
    Document(page_content="قوالب الأوامر Prompt Templates وسلاسل LCEL تربط مكونات لانج تشين بطريقة تركيبية منظمة."),
    Document(page_content="رقم المنفذ الافتراضي لقاعدة بيانات بوستجريس هو 5432.")
]

vector_store = FAISS.from_documents(docs, embeddings)
print("تم بناء الفهرس بنجاح بعدد وثائق:", vector_store.index.ntotal)

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_28664\3490644514.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء الفهرس بنجاح بعدد وثائق: 7


### 2. سيناريو (أ): السؤال الاستكشافي العام - تفوق MMR الكاسح
تعال نسأل سؤال عام: 'ما هي قدرات وإمكانيات لانج تشين؟' ونقارن بين البحث العادي ومسترجع MMR.

In [2]:
query_broad = "ما هي قدرات وإمكانيات لانج تشين؟"

# 1. البحث العادي (Similarity)
sim_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
sim_results = sim_retriever.invoke(query_broad)

print("=== نتائج البحث العادي (لاحظ التكرار في FAISS) ===")
for i, d in enumerate(sim_results, 1):
    print(f"[{i}]: {d.page_content}")

# 2. مسترجع أقصى تنوع (MMR)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5}
)
mmr_results = mmr_retriever.invoke(query_broad)

print("\n=== نتائج مسترجع MMR (تنوع وثراء سياقي مذهل) ===")
for i, d in enumerate(mmr_results, 1):
    print(f"[{i}]: {d.page_content}")

=== نتائج البحث العادي (لاحظ التكرار في FAISS) ===
[1]: فهارس FAISS في لانج تشين تسرع البحث الدلالي في المشاريع الكبرى.
[2]: قوالب الأوامر Prompt Templates وسلاسل LCEL تربط مكونات لانج تشين بطريقة تركيبية منظمة.
[3]: إدارة الذاكرة Memory في LangChain تمكن الشات بوت من تذكر سياق الحوار عبر الأدوار المختلفة.

=== نتائج مسترجع MMR (تنوع وثراء سياقي مذهل) ===
[1]: فهارس FAISS في لانج تشين تسرع البحث الدلالي في المشاريع الكبرى.
[2]: رقم المنفذ الافتراضي لقاعدة بيانات بوستجريس هو 5432.
[3]: قوالب الأوامر Prompt Templates وسلاسل LCEL تربط مكونات لانج تشين بطريقة تركيبية منظمة.


### 3. سيناريو (ب): السؤال عن حقيقة مفردة محددة - متى نتجنب MMR؟
دلوقتي المستخدم بيسأل عن حقيقة محددة بدقة: 'ما هو رقم المنفذ الافتراضي لقاعدة بيانات بوستجريس؟'
هنا إجبار النظام على التنوع غلط، لأن مفيش غير معلومة واحدة صحيحة!

In [3]:
query_fact = "ما هو رقم المنفذ الافتراضي لقاعدة بيانات بوستجريس؟"

res_sim = sim_retriever.invoke(query_fact)
print("البحث العادي (دقة مباشرة 100%):")
print(res_sim[0].page_content)

البحث العادي (دقة مباشرة 100%):
رقم المنفذ الافتراضي لقاعدة بيانات بوستجريس هو 5432.


### 4. جرب بنفسك!
1. جرب تسأل عن 'كيف يدعم لانج تشين إدارة الذاكرة والوكلاء؟' وشوف إزاي MMR بيختار المقطعين مع بعض.
2. غير قيمة lambda_mult وخليها 0.1 وشوف هل التنوع زاد أكتر؟
3. لما تكون شغال على نظام أسئلة شائعة FAQ، هل تنصح باستخدام MMR أم البحث العادي؟ ولماذا؟